# Day 8 Tutorial：数据划分协议与测试集边界

## Goal

重新加载 ESOL scaffold splits，核对 shape、特征有限值和精确 ID 重叠；不读取 test 标签，不训练模型，不产生 test 预测。

## Setup

显式使用 1024 维 ECFP、原始 logS 和仓库内缓存。

In [1]:
import contextlib
import io
from pathlib import Path

import deepchem as dc
import numpy as np
import pandas as pd
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.warning')

def find_repo_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('请从 ML-Learning 仓库内部运行')

repo_root = find_repo_root()
data_dir = repo_root / '.cache' / 'deepchem'
data_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_root.name)
print('cache ready:', data_dir.exists())

Skipped loading some Pytorch utilities, missing a dependency. No module named 'torch'


This module requires PyTorch to be installed.


No normalization for SPS. Feature removed!


No normalization for AvgIpc. Feature removed!


No normalization for NumAmideBonds. Feature removed!


...

Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch'


Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'torch'


Skipped loading some Jax models, missing a dependency. No module named 'jax'


Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


repository: ML-Learning
cache ready: True


In [2]:
featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)
captured_stdout = io.StringIO()
captured_stderr = io.StringIO()
with contextlib.redirect_stdout(captured_stdout), contextlib.redirect_stderr(captured_stderr):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer=featurizer,
        splitter='scaffold',
        transformers=[],
        reload=True,
        data_dir=str(data_dir),
        save_dir=str(data_dir),
    )

train_dataset, valid_dataset, test_dataset = datasets
assert transformers == []
print('task:', tasks)
print('split lengths:', [len(dataset) for dataset in datasets])

task: ['measured log solubility in mols per litre']
split lengths: [902, 113, 113]


## Steps

建立包含用途的 split 表，并计算三个精确 ID 交集。train/validation 可以核对标签完整性，test 只核对 X、样本数和 ID。

In [3]:
split_datasets = {
    'train': train_dataset,
    'valid': valid_dataset,
    'test': test_dataset,
}
split_purposes = {
    'train': '允许 fit 模型和预处理器',
    'valid': '开发阶段比较；不允许 fit',
    'test': '历史已暴露；Day 8 不预测、不选参',
}
rows = []

for split_name, dataset in split_datasets.items():
    X = np.asarray(dataset.X)
    ids = np.asarray(dataset.ids)
    may_inspect_labels = split_name != 'test'
    y = np.asarray(dataset.y).reshape(-1) if may_inspect_labels else None
    rows.append({
        'split': split_name,
        'n_samples': int(X.shape[0]),
        'n_features': int(X.shape[1]),
        'n_ids': int(ids.shape[0]),
        'n_labels_checked': int(y.shape[0]) if may_inspect_labels else None,
        'all_X_finite': bool(np.isfinite(X).all()),
        'all_y_finite': bool(np.isfinite(y).all()) if may_inspect_labels else None,
        'label_policy': '允许完整性检查' if may_inspect_labels else '未读取',
        'purpose': split_purposes[split_name],
    })

split_table = pd.DataFrame(rows)
split_table

,split,n_samples,n_features,n_ids,n_labels_checked,all_X_finite,all_y_finite,label_policy,purpose
0,train,902,1024,902,902.0,True,True,允许完整性检查,允许 fit 模型和预处理器
1,valid,113,1024,113,113.0,True,True,允许完整性检查,开发阶段比较；不允许 fit
2,test,113,1024,113,NaN,True,None,未读取,历史已暴露；Day 8 不预测、不选参


In [4]:
id_sets = {
    name: set(map(str, dataset.ids))
    for name, dataset in split_datasets.items()
}
overlap_table = pd.DataFrame([
    {'pair': 'train_valid', 'exact_id_overlap': len(id_sets['train'] & id_sets['valid'])},
    {'pair': 'train_test', 'exact_id_overlap': len(id_sets['train'] & id_sets['test'])},
    {'pair': 'valid_test', 'exact_id_overlap': len(id_sets['valid'] & id_sets['test'])},
])
overlap_table

,pair,exact_id_overlap
0,train_valid,0
1,train_test,0
2,valid_test,0


## Checks

检查当前固定协议；精确 ID 无重叠不等于所有分子骨架完全独立。

In [5]:
expected_sizes = {'train': 902, 'valid': 113, 'test': 113}
assert split_table['n_features'].eq(1024).all()
assert split_table['all_X_finite'].all()
assert split_table.loc[
    split_table['split'].isin(['train', 'valid']),
    'all_y_finite',
].all()
for row in split_table.to_dict(orient='records'):
    assert row['n_samples'] == expected_sizes[row['split']]
    assert row['n_samples'] == row['n_ids']
    if row['split'] != 'test':
        assert row['n_samples'] == row['n_labels_checked']
    else:
        assert pd.isna(row['n_labels_checked'])
assert overlap_table['exact_id_overlap'].eq(0).all()
print('split contract checks passed')
print('test labels accessed: no')
print('test predictions created: no')

split contract checks passed
test labels accessed: no
test predictions created: no


## Next Steps

独立完成 `03_exercises.md`，在个人副本中保存 split 与 overlap 表及自己的 test 政策；不要生成 test 预测或指标。